In [ ]:
import pandas as pd
import requests
import keyring
import urllib3
from pyodata import Client
from datetime import datetime

# =================================================================
# 1. Base Configuration & Security Settings
# =================================================================
SERVICE_URL = 'https://yousap/sap/opu/odata/sap/ZMTEST5_UMD_SRV/'
SAP_USER = "sap userid"
SERVICE_NAME = "sap_100"

# Disable SSL warnings (for SAP self-signed certificates)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

def generate_sap_report():
    try:
        print("Connecting to SAP and fetching data...")
        
        # 2. Securely retrieve password and establish session
        pwd = keyring.get_password(SERVICE_NAME, SAP_USER)
        if not pwd:
            raise ValueError(f"Password for service {SERVICE_NAME} not found in keyring. Please set the password first.")

        session = requests.Session()
        session.auth = (SAP_USER, pwd)
        session.verify = False
        session.timeout = 15

        # 3. Initialize pyodata client and fetch data
        sap_client = Client(SERVICE_URL, session)
        # Fetch all airline entities
        entities = sap_client.entity_sets.scarrSet.get_entities().execute()
        
        # 4. Convert data to DataFrame for processing
        data = []
        for entity in entities:
            data.append({
                "ID": entity.Carrid,
                "Name": entity.Carrname,
                "Currency": entity.Currcode,
                "Url": entity.Url if entity.Url else "#"
            })
        df = pd.DataFrame(data)

        # =================================================================
        # 5. Build Minimalist Luxury HTML Template (Inline CSS)
        # =================================================================
        now_str = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        
        html_content = f"""
        <!DOCTYPE html>
        <html lang="en">
        <head>
            <meta charset="UTF-8">
            <style>
                :root {{
                    --sap-blue: #0070d2;
                    --text-main: #32363a;
                    --text-sub: #6a6d70;
                    --bg-light: #f3f4f5;
                    --white: #ffffff;
                }}
                
                body {{ 
                    background-color: var(--bg-light); 
                    font-family: 'Segoe UI', Arial, sans-serif; 
                    margin: 0; padding: 40px 20px;
                    color: var(--text-main);
                }}

                .report-card {{
                    max-width: 1000px;
                    margin: 0 auto;
                    background: var(--white);
                    border-radius: 12px;
                    box-shadow: 0 8px 24px rgba(0,0,0,0.08);
                    overflow: hidden;
                }}

                .header {{
                    padding: 30px;
                    background: linear-gradient(135deg, #004487 0%, #0070d2 100%);
                    color: white;
                }}

                .header h1 {{ margin: 0; font-size: 24px; font-weight: 500; letter-spacing: 1px; }}
                .header .info {{ margin-top: 10px; font-size: 13px; opacity: 0.8; line-height: 1.6; }}

                .content {{ padding: 20px; }}

                table {{ 
                    width: 100%; border-collapse: collapse; margin-top: 10px; 
                }}
                
                th {{ 
                    text-align: left; padding: 16px; 
                    background-color: #fafafa;
                    color: var(--text-sub); font-weight: 600;
                    font-size: 14px; border-bottom: 2px solid #eeeeee;
                }}

                td {{ 
                    padding: 16px; border-bottom: 1px solid #f0f0f0; 
                    font-size: 14px; vertical-align: middle;
                }}

                tr:last-child td {{ border-bottom: none; }}
                tr:hover {{ background-color: #f8fbff; }}

                .id-badge {{
                    background: #e1f0ff; color: #0056b3;
                    padding: 4px 10px; border-radius: 6px;
                    font-family: 'Consolas', monospace; font-weight: bold;
                }}

                .curr-tag {{
                    background: #f0f0f0; color: #555;
                    padding: 2px 8px; border-radius: 4px; font-size: 12px;
                }}

                .btn-link {{
                    display: inline-block;
                    text-decoration: none; color: white;
                    background: var(--sap-blue);
                    padding: 6px 16px; border-radius: 20px;
                    font-size: 12px; transition: all 0.3s;
                }}

                .btn-link:hover {{ background: #005fb2; transform: translateY(-1px); box-shadow: 0 4px 8px rgba(0,0,0,0.15); }}
                
                .footer {{
                    text-align: center; padding: 20px; 
                    font-size: 12px; color: var(--text-sub);
                    background: #fafafa; border-top: 1px solid #eee;
                }}
            </style>
        </head>
        <body>
            <div class="report-card">
                <div class="header">
                    <h1>✈️ Airline Master Data Center</h1>
                    <div class="info">
                        <strong>Service URL:</strong> {SERVICE_URL}<br>
                        <strong>Extraction Time:</strong> {now_str} | <strong>User:</strong> {SAP_USER}
                    </div>
                </div>
                
                <div class="content">
                    <table>
                        <thead>
                            <tr>
                                <th width="15%">Airline ID</th>
                                <th width="45%">Airline Full Name</th>
                                <th width="15%">Local Currency</th>
                                <th width="25%">Official Link</th>
                            </tr>
                        </thead>
                        <tbody>
        """

        # Populate data rows
        for _, row in df.iterrows():
            html_content += f"""
                            <tr>
                                <td><span class="id-badge">{row['ID']}</span></td>
                                <td style="font-weight: 500;">{row['Name']}</td>
                                <td><span class="curr-tag">{row['Currency']}</span></td>
                                <td><a href="{row['Url']}" target="_blank" class="btn-link">View Details</a></td>
                            </tr>
            """

        html_content += f"""
                        </tbody>
                    </table>
                </div>
                <div class="footer">
                    © {datetime.now().year} Digital Supply Chain Reporting System - Auto Generated
                </div>
            </div>
        </body>
        </html>
        """

        # 6. Save report file
        filename = "SAP_Airline_Report.html"
        with open(filename, "w", encoding="utf-8") as f:
            f.write(html_content)
        
        print(f"✨ Report generated successfully! Please check: {filename}")

    except Exception as e:
        print(f"❌ Execution failed: {e}")

if __name__ == "__main__":
    generate_sap_report()